# Complete Carry Strategy for Futures - End-to-End Implementation

This notebook demonstrates a complete carry trading strategy for interest rate futures using the ARBS framework.

**What You'll Learn**:
1. What carry trading is and why it works
2. How to calculate carry signals from calendar spreads
3. How to convert signals to alphas using IC × Vol × Z
4. How to optimize portfolios with mean-variance
5. How to backtest and analyze performance

**Pipeline**:
```
Market Data → Carry Signals → Alpha Generation → Risk Model → Optimization → Backtest → Analysis
```

---

## Part 1: What is Carry Trading?

### Economic Rationale

**Carry** is the expected return from holding an asset, assuming no price changes.

For futures contracts:
- **Contango**: Back contract > Front contract (negative carry)
  - Example: Front = 94.50, Back = 94.45 → Carry = -5 bps
  - Holding the front contract loses value as it converges to back
  
- **Backwardation**: Front contract > Back contract (positive carry)
  - Example: Front = 94.50, Back = 94.55 → Carry = +5 bps
  - Holding the front contract gains value as it converges to back

### Why Does Carry Work?

1. **Risk Premium**: Investors demand compensation for holding positions
2. **Mean Reversion**: Calendar spreads tend to revert to fair value
3. **Time Decay**: Positive carry accrues over time

### Historical Performance

From academic research:
- **Information Coefficient (IC)**: 0.05 - 0.10 (good to very good)
- **Sharpe Ratio**: 1.0 - 1.5 (for well-constructed carry strategies)
- **Half-life**: 60-90 days (medium frequency signal)

### Strategy Hypothesis

**"Contracts with higher carry will outperform contracts with lower carry over time."**

We test this by:
1. Calculating carry for each contract (annualized bps/year)
2. Converting carry to z-scores (standardized signals)
3. Building a portfolio that is long high-carry, short low-carry
4. Measuring performance (IC, Sharpe, returns)

---

## Part 2: Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta
from typing import Dict, List

# ARBS components
from Signals.Futures.CarrySignal import CarrySignal
from Signals.AlphaGenerator import AlphaGenerator
from Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkage
from Risk.Volatility.RealizedVolatility import RealizedVolatility
from Risk.Returns.ReturnsCalculator import ReturnsCalculator
from Optimizer.MeanVarianceOptimizer import MeanVarianceOptimizer
from Backtest.Backtest import Backtest
from Analysis.TearSheet import TearSheet
from Signals.Utils.IC import calculate_ic

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Setup complete!")

---

## Part 3: Generate Mock Futures Data

We'll create synthetic SOFR futures data with realistic properties:
- 8 contracts across 2 years (quarterly expiries)
- Realistic carry structure (contango/backwardation)
- Time-varying curve slopes (regime changes)
- Market noise (realistic volatility)

In [ ]:
class RealisticMockMDP:
    """
    Mock market data provider with realistic carry structure.
    
    Features:
    - Time-varying curve slopes (regime changes)
    - Mean-reverting calendar spreads
    - Realistic volatility levels
    - Both contango and backwardation regimes
    """
    
    def __init__(self, base_rate: float = 5.0, vol: float = 0.02):
        self.base_rate = base_rate
        self.vol = vol
        
        # Generate time-varying carry structure
        # Simulate 3 regimes: steep contango, flat, backwardation
        self.regime_history = []
        
    def set_regime(self, as_of: date, n_days_elapsed: int):
        """
        Set carry regime based on time.
        
        Regimes cycle through:
        - Days 0-100: Steep contango (+15 bps/quarter)
        - Days 101-200: Flat curve (+2 bps/quarter)
        - Days 201-300: Backwardation (-8 bps/quarter)
        - Days 301+: Return to contango
        """
        if n_days_elapsed < 100:
            return 0.15  # Steep contango
        elif n_days_elapsed < 200:
            return 0.02  # Flat
        elif n_days_elapsed < 300:
            return -0.08  # Backwardation
        else:
            return 0.10  # Moderate contango
    
    def get_pricer(self, currency: str, as_of: date):
        return self
    
    def futures_price(self, contract: str, as_of: date = None, n_days_elapsed: int = 0) -> float:
        """
        Generate realistic futures price with carry structure.
        
        Price = 100 - (base_rate + carry × quarters_out + noise)
        """
        # Extract quarter from contract code
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        if len(contract) < 2:
            return 100.0 - self.base_rate
        
        quarter_code = contract[-2]
        quarter = quarter_map.get(quarter_code, 0)
        
        # Get regime-specific carry
        carry_spread = self.set_regime(as_of, n_days_elapsed)
        
        # Calculate implied rate
        rate = self.base_rate + (quarter * carry_spread)
        
        # Add mean-reverting noise
        noise = np.random.normal(0, self.vol)
        
        # Convert to futures price (100 - rate)
        price = 100.0 - rate + noise
        
        return price


# Define contract universe (8 SOFR futures, quarterly)
contracts = [
    'SFRZ4',  # Dec 2024
    'SFRH5',  # Mar 2025
    'SFRM5',  # Jun 2025
    'SFRU5',  # Sep 2025
    'SFRZ5',  # Dec 2025
    'SFRH6',  # Mar 2026
    'SFRM6',  # Jun 2026
    'SFRU6',  # Sep 2026
]

print(f"Universe: {len(contracts)} SOFR futures contracts")
print(f"Contracts: {', '.join(contracts)}")

# Create market data provider
mdp = RealisticMockMDP(base_rate=5.0, vol=0.02)

print("\n✓ Mock market data provider created")
print("  • Base rate: 5.0%")
print("  • Volatility: 2% (realistic for SOFR)")
print("  • Regime changes: Yes (contango → flat → backwardation)")

In [ ]:
# Generate sample prices to visualize curve structure
sample_date = date(2024, 1, 1)

# Show curve at different regimes
regimes = [
    (0, "Steep Contango (Days 0-100)"),
    (150, "Flat Curve (Days 101-200)"),
    (250, "Backwardation (Days 201-300)"),
    (350, "Moderate Contango (Days 301+)")
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (n_days, regime_name) in enumerate(regimes):
    prices = [mdp.futures_price(c, sample_date, n_days) for c in contracts]
    
    # Calculate calendar spreads (carry)
    spreads = [prices[i] - prices[i+1] for i in range(len(prices)-1)]
    
    # Plot prices
    axes[idx].plot(range(len(contracts)), prices, marker='o', linewidth=2, markersize=8)
    axes[idx].set_title(regime_name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Contract (Front → Back)', fontsize=10)
    axes[idx].set_ylabel('Futures Price', fontsize=10)
    axes[idx].set_xticks(range(len(contracts)))
    axes[idx].set_xticklabels(contracts, rotation=45)
    axes[idx].grid(True, alpha=0.3)
    
    # Add annotation about average spread
    avg_spread = np.mean(spreads)
    spread_type = "Backwardation" if avg_spread > 0 else "Contango"
    axes[idx].text(0.05, 0.95, f"Avg spread: {avg_spread:.3f} ({spread_type})",
                   transform=axes[idx].transAxes, fontsize=9,
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("  • Upward sloping = Contango = Negative carry")
print("  • Downward sloping = Backwardation = Positive carry")
print("  • Flat = Neutral carry")
print("\nStrategy will be long backwardation, short contango")

---

## Part 4: Calculate Carry Signals

We'll use `CarrySignal` to calculate annualized carry from calendar spreads.

**Formula**:
```
Carry (bps/year) = (Front Price - Back Price) / Days to Roll × 10,000 × 252
```

The signal is then standardized to z-scores (mean=0, std=1) for portfolio construction.

In [ ]:
# Initialize carry signal calculator
carry_signal = CarrySignal(
    name='futures_carry',
    standardize=True,  # Convert to z-scores
    annualize=True,    # Return annualized carry (bps/year)
    include_basis=False,  # Don't include futures-swap basis adjustment
    business_days_per_year=252
)

print("✓ Carry signal calculator initialized")
print(f"  • Signal name: {carry_signal.name}")
print(f"  • Standardize: {carry_signal.standardize}")
print(f"  • Annualize: {carry_signal.annualize}")

# Let's manually calculate carry for one point in time to understand the mechanics
sample_date = date(2024, 1, 1)
n_days = 50  # Steep contango regime

print("\n" + "="*70)
print("Manual Carry Calculation Example (Steep Contango Regime)")
print("="*70)

# Get prices for all contracts
prices = {c: mdp.futures_price(c, sample_date, n_days) for c in contracts}

# Calculate carry for each adjacent pair
carry_data = []
for i in range(len(contracts) - 1):
    front = contracts[i]
    back = contracts[i+1]
    
    front_price = prices[front]
    back_price = prices[back]
    
    # Calendar spread (front - back)
    calendar_spread = front_price - back_price
    
    # Assume 90 days between contracts (quarterly)
    days_to_roll = 90
    
    # Annualized carry (bps/year)
    carry_bps = (calendar_spread / days_to_roll) * 10000 * 252
    
    carry_data.append({
        'contract': front,
        'front_price': front_price,
        'back_price': back_price,
        'calendar_spread': calendar_spread,
        'carry_bps_year': carry_bps
    })

carry_df = pd.DataFrame(carry_data)
print("\n", carry_df.to_string(index=False))

print("\n💡 Interpretation:")
print("  • Positive carry: Front > Back (backwardation) → Buy")
print("  • Negative carry: Front < Back (contango) → Sell")
print(f"  • Average carry: {carry_df['carry_bps_year'].mean():.1f} bps/year")

---

## Part 5: Alpha Generation (IC × Vol × Z)

Raw carry signals are **z-scores** (dimensionless).

We need to convert them to **expected returns** (alphas) using the Grinold-Kahn formula:

$$\alpha_i = IC \times \sigma_i \times z_i$$

Where:
- $\alpha_i$ = expected return for asset $i$
- $IC$ = Information Coefficient (forecasting skill, typically 0.05-0.10)
- $\sigma_i$ = volatility of asset $i$ (annualized)
- $z_i$ = signal z-score

**Why is this critical?**

Without this conversion:
- Z-score = 2.0 → optimizer treats as 200% expected return (absurd!)

With conversion (IC=0.05, Vol=15%):
- Z-score = 2.0 → $\alpha$ = 0.05 × 0.15 × 2.0 = 0.015 = 1.5% (sensible!)

In [ ]:
# Initialize alpha generator
alpha_generator = AlphaGenerator(
    IC=0.05,  # 5% IC (typical for carry strategies)
    vol_estimator=RealizedVolatility(lookback=60, annualization_factor=252)
)

print("✓ Alpha generator initialized")
print(f"  • IC (Information Coefficient): {alpha_generator.IC}")
print(f"  • Volatility estimator: RealizedVolatility (60-day lookback)")

# Demonstrate alpha generation with example
print("\n" + "="*70)
print("Example: Converting Z-Scores to Alphas")
print("="*70)

# Mock signal z-scores
example_signals = {
    'SFRZ4': 2.0,   # Strong positive signal
    'SFRH5': 1.0,   # Moderate positive signal
    'SFRM5': 0.0,   # Neutral signal
    'SFRU5': -1.0,  # Moderate negative signal
    'SFRZ5': -2.0,  # Strong negative signal
}

# Mock volatilities (15% annualized, typical for SOFR futures)
example_vols = {c: 0.15 for c in example_signals.keys()}

# Calculate alphas manually
IC = 0.05
alpha_examples = []
for contract, z_score in example_signals.items():
    vol = example_vols[contract]
    alpha = IC * vol * z_score
    alpha_examples.append({
        'Contract': contract,
        'Z-Score': z_score,
        'Volatility': f"{vol:.1%}",
        'Alpha': f"{alpha:.2%}",
        'Interpretation': 'Strong Buy' if z_score >= 1.5 else 
                         'Buy' if z_score >= 0.5 else
                         'Neutral' if abs(z_score) < 0.5 else
                         'Sell' if z_score <= -0.5 else 'Strong Sell'
    })

alpha_df = pd.DataFrame(alpha_examples)
print("\n", alpha_df.to_string(index=False))

print("\n💡 Key Insight:")
print("  • Alphas are scaled to be realistic expected returns (1-2%)")
print("  • Higher volatility assets get larger alphas (for same z-score)")
print("  • IC scales all alphas by forecasting skill")
print("\nThis prevents optimizer from taking absurdly large positions!")

---

## Part 6: Portfolio Optimization

We use **Mean-Variance Optimization** (Markowitz 1952) to find optimal portfolio weights.

**Objective**:
$$\max_w \quad \alpha'w - \frac{\lambda}{2} w'\Sigma w$$

Subject to:
- $\sum_i w_i = 1$ (fully invested)
- $w_i \geq 0$ (long-only, optional)

Where:
- $\alpha$ = vector of expected returns (alphas)
- $w$ = vector of portfolio weights
- $\Sigma$ = covariance matrix
- $\lambda$ = risk aversion (higher = more conservative)

**Risk Model**: We use Ledoit-Wolf shrinkage to estimate covariance, which is more robust than sample covariance for small samples.

In [ ]:
# Initialize optimizer
optimizer = MeanVarianceOptimizer(
    risk_aversion=1.0,  # Moderate risk aversion
    long_only=True,     # Only long positions (for simplicity)
)

print("✓ Mean-Variance Optimizer initialized")
print(f"  • Risk aversion (λ): {optimizer.risk_aversion}")
print(f"  • Long only: {optimizer.long_only}")
print(f"  • Method: Quadratic programming (scipy.optimize)")

# Initialize risk model (covariance estimator)
risk_model = LedoitWolfShrinkage()

print("\n✓ Risk model initialized")
print("  • Method: Ledoit-Wolf shrinkage")
print("  • Advantage: Robust to estimation error vs sample covariance")
print("  • Works well when T ≈ N (time periods ≈ number of assets)")

---

## Part 7: Run Complete Backtest

Now we'll run the full pipeline using `Backtest`:

1. For each date:
   - Get prices via adapter
   - Calculate carry signals (z-scores)
   - Convert to alphas (IC × Vol × Z)
   - Estimate covariance from return history
   - Optimize portfolio weights
   - Track positions and calculate P&L

2. Calculate performance metrics:
   - Sharpe ratio
   - Information Coefficient (IC)
   - Total return
   - Drawdowns

In [ ]:
# Define backtest period (1 year, weekly rebalancing)
start_date = date(2024, 1, 1)
end_date = date(2024, 12, 31)
dates = pd.date_range(start_date, end_date, freq='W-MON').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

print(f"Backtest Period: {start_date} to {end_date}")
print(f"Rebalancing: Weekly (every Monday)")
print(f"Number of periods: {len(dates)}")
print(f"Universe: {len(contracts)} contracts")

# Initialize backtest
backtest = Backtest(
    mdp=mdp,
    risk_aversion=1.0,
    long_only=True,
    min_history=10,  # Need 10 weeks before estimating covariance
    IC=0.05          # 5% IC assumption
)

print("\n" + "="*70)
print("Running Backtest...")
print("="*70)
print("This will:")
print("  1. Generate prices for each date")
print("  2. Calculate carry signals")
print("  3. Convert to alphas (IC × Vol × Z)")
print("  4. Estimate covariance matrix")
print("  5. Optimize portfolio weights")
print("  6. Track P&L and returns")
print()

# Run backtest
result = backtest.run(contracts=contracts, dates=dates)

print("\n✓ Backtest complete!")
print(f"  • Periods executed: {len(result.returns)}")
print(f"  • Assets tracked: {len(result.weights.columns)}")

---

## Part 8: Performance Analysis with TearSheet

We'll use `TearSheet` to calculate comprehensive performance metrics:

- **Sharpe Ratio**: Risk-adjusted return (annualized)
- **Information Coefficient (IC)**: Correlation between forecasts and returns
- **Total Return**: Cumulative return over backtest period
- **Max Drawdown**: Largest peak-to-trough decline
- **Calmar Ratio**: Annual return / Max drawdown
- **Volatility**: Annualized standard deviation

In [ ]:
# Create tear sheet analyzer
tear_sheet = TearSheet(
    returns=pl.Series(result.returns.values, dtype=pl.Float64),
    periods_per_year=52,  # Weekly returns
    risk_free_rate=0.05   # 5% risk-free rate
)

# Calculate metrics
metrics = tear_sheet.calculate_metrics()

print("="*70)
print("PERFORMANCE METRICS")
print("="*70)
print()
print(metrics)
print()

# Additional backtest-specific metrics
print("="*70)
print("STRATEGY-SPECIFIC METRICS")
print("="*70)
print()
print(f"Information Coefficient (IC): {result.ic:>8.3f}")
print(f"  • Measures signal quality (correlation of forecast vs realized)")
print(f"  • Typical range: 0.02 - 0.10 for quant strategies")
print(f"  • IC > 0.05 = Good signal quality")
print()
print(f"Total Return:                 {result.total_return:>8.2%}")
print(f"Number of Periods:            {len(result.returns):>8}")
print(f"Number of Assets:             {len(result.weights.columns):>8}")
print()

# Calculate turnover
weight_changes = result.weights.diff().abs().sum(axis=1)
avg_turnover = weight_changes.mean()
print(f"Average Weekly Turnover:      {avg_turnover:>8.2%}")
print(f"Annualized Turnover:          {avg_turnover * 52:>8.1%}")
print()

print("="*70)
print("PERFORMANCE INTERPRETATION")
print("="*70)
print()

# Interpret results
if metrics.sharpe_ratio > 1.0:
    print("✓ Sharpe > 1.0: Good risk-adjusted returns")
elif metrics.sharpe_ratio > 0.5:
    print("✓ Sharpe > 0.5: Acceptable risk-adjusted returns")
else:
    print("⚠ Sharpe < 0.5: Suboptimal risk-adjusted returns")

if result.ic > 0.05:
    print("✓ IC > 0.05: Strong signal quality")
elif result.ic > 0.02:
    print("✓ IC > 0.02: Acceptable signal quality")
else:
    print("⚠ IC < 0.02: Weak signal quality")

if abs(metrics.max_drawdown) < 0.10:
    print("✓ Max Drawdown < 10%: Low downside risk")
elif abs(metrics.max_drawdown) < 0.20:
    print("✓ Max Drawdown < 20%: Moderate downside risk")
else:
    print("⚠ Max Drawdown > 20%: High downside risk")

print()
print("💡 Remember: Negative results are OK for MVP!")
print("   Goal is ACCURATE MEASUREMENT, not necessarily profitability.")

---

## Part 9: Visualization - Cumulative Returns

In [ ]:
# Calculate cumulative returns
cumulative_returns = tear_sheet.calculate_cumulative_returns()

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 1. Cumulative returns
axes[0].plot(result.returns.index, (1 + cumulative_returns.to_numpy()), 
             linewidth=2.5, color='steelblue', label='Strategy')
axes[0].axhline(y=1.0, color='black', linestyle='--', alpha=0.3, linewidth=1)
axes[0].set_title('Cumulative Returns (Growth of $1)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Portfolio Value ($)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Add annotations for key points
final_value = 1 + cumulative_returns.to_numpy()[-1]
axes[0].annotate(f'Final: ${final_value:.3f}',
                xy=(result.returns.index[-1], final_value),
                xytext=(10, 0), textcoords='offset points',
                fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.5))

# 2. Drawdowns
drawdowns = tear_sheet.calculate_drawdowns()
axes[1].fill_between(result.returns.index, drawdowns.to_numpy(), 0, 
                      color='red', alpha=0.3, label='Drawdown')
axes[1].set_title('Drawdowns (Distance from Peak)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Drawdown', fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Add max drawdown annotation
max_dd_idx = drawdowns.arg_min()
max_dd_date = result.returns.index[max_dd_idx]
max_dd_value = drawdowns[max_dd_idx]
axes[1].annotate(f'Max DD: {max_dd_value:.2%}',
                xy=(max_dd_date, max_dd_value),
                xytext=(0, -20), textcoords='offset points',
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='red'))

plt.tight_layout()
plt.show()

print("\n💡 Chart Interpretation:")
print("  • Top chart: Shows cumulative wealth (start with $1)")
print("  • Bottom chart: Shows underwater periods (drawdowns)")
print(f"  • Final portfolio value: ${final_value:.3f}")
print(f"  • Maximum drawdown: {metrics.max_drawdown:.2%}")

---

## Part 10: Visualization - Signal Distribution and Portfolio Weights

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of weekly returns
axes[0, 0].hist(result.returns, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0, 0].axvline(result.returns.mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {result.returns.mean():.3%}')
axes[0, 0].set_title('Distribution of Weekly Returns', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Weekly Return', fontsize=10)
axes[0, 0].set_ylabel('Frequency', fontsize=10)
axes[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Average portfolio weights
avg_weights = result.weights.mean().sort_values(ascending=False)
axes[0, 1].barh(avg_weights.index, avg_weights.values, color='green', alpha=0.7)
axes[0, 1].set_title('Average Portfolio Weights', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Weight', fontsize=10)
axes[0, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0, 1].grid(True, alpha=0.3, axis='x')

# 3. Weights over time (heatmap)
weights_matrix = result.weights.T
im = axes[1, 0].imshow(weights_matrix, aspect='auto', cmap='RdYlGn', 
                       vmin=0, vmax=weights_matrix.max().max())
axes[1, 0].set_title('Portfolio Weights Over Time', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Time Period', fontsize=10)
axes[1, 0].set_ylabel('Contract', fontsize=10)
axes[1, 0].set_yticks(range(len(contracts)))
axes[1, 0].set_yticklabels(contracts)
plt.colorbar(im, ax=axes[1, 0], label='Weight')

# 4. Rolling Sharpe ratio (60-day window)
rolling_window = 12  # 12 weeks ≈ 3 months
rolling_sharpe = result.returns.rolling(rolling_window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)
axes[1, 1].plot(result.returns.index, rolling_sharpe, linewidth=2, color='purple')
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
axes[1, 1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, linewidth=1, 
                   label='Sharpe = 1.0')
axes[1, 1].set_title(f'Rolling Sharpe Ratio ({rolling_window}-week)', 
                     fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Date', fontsize=10)
axes[1, 1].set_ylabel('Sharpe Ratio', fontsize=10)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Chart Interpretation:")
print("  • Top-left: Return distribution shows strategy risk/reward profile")
print("  • Top-right: Average weights show which contracts are favored")
print("  • Bottom-left: Heatmap shows how allocation shifts over time")
print("  • Bottom-right: Rolling Sharpe shows time-varying performance")

---

## Part 11: Deep Dive - Signal Quality Analysis

Let's analyze the relationship between carry signals and subsequent returns (Information Coefficient).

In [ ]:
# For IC analysis, we need signals and forward returns
# We'll extract this from the backtest history

print("="*70)
print("SIGNAL QUALITY ANALYSIS")
print("="*70)
print()
print("Analyzing relationship between carry signals and realized returns...")
print()

# Calculate IC statistics
print(f"Information Coefficient (IC): {result.ic:.4f}")
print()
print("IC Interpretation:")
print(f"  • IC = {result.ic:.4f} means carry signals have a {abs(result.ic):.2%} correlation")
print("    with next-period returns")
print()

if result.ic > 0.10:
    print("  ⭐ EXCELLENT: IC > 0.10 is very rare and indicates strong forecasting skill")
elif result.ic > 0.05:
    print("  ✓ GOOD: IC > 0.05 is typical for successful quant strategies")
elif result.ic > 0.02:
    print("  ✓ ACCEPTABLE: IC > 0.02 can still be profitable with high breadth")
elif result.ic > 0:
    print("  ⚠ WEAK: IC > 0 but < 0.02 indicates minimal forecasting skill")
else:
    print("  ❌ NEGATIVE: IC < 0 means signal is inversely related to returns")
    print("     (Strategy should flip signs or abandon signal)")

print()
print("Fundamental Law of Active Management:")
print(f"  IR = IC × √BR")
print()
print(f"  IC (Information Coefficient):  {result.ic:.4f}")
print(f"  BR (Breadth): ~{len(contracts) * len(dates):.0f} (contracts × periods)")
print(f"  Theoretical IR: {result.ic * np.sqrt(len(contracts) * len(dates)):.3f}")
print(f"  Actual Sharpe: {metrics.sharpe_ratio:.3f}")
print()
print("Note: Actual Sharpe differs from theoretical IR due to:")
print("  • Correlation between assets (reduces effective breadth)")
print("  • Transaction costs (not modeled in this example)")
print("  • Constraint effects (long-only reduces transfer coefficient)")

---

## Part 12: Sensitivity Analysis - Risk Aversion

How does risk aversion affect portfolio performance?

In [ ]:
# Test different risk aversion levels
risk_aversions = [0.5, 1.0, 2.0, 4.0]
sensitivity_results = []

print("="*70)
print("SENSITIVITY ANALYSIS: Risk Aversion")
print("="*70)
print()
print("Testing how risk aversion affects performance...")
print()

for ra in risk_aversions:
    # Create backtest with different risk aversion
    test_backtest = Backtest(
        mdp=mdp,
        risk_aversion=ra,
        long_only=True,
        min_history=10,
        IC=0.05
    )
    
    # Run backtest
    np.random.seed(42)  # Same seed for fair comparison
    test_result = test_backtest.run(contracts=contracts, dates=dates)
    
    # Calculate metrics
    test_tear_sheet = TearSheet(
        returns=pl.Series(test_result.returns.values, dtype=pl.Float64),
        periods_per_year=52,
        risk_free_rate=0.05
    )
    test_metrics = test_tear_sheet.calculate_metrics()
    
    sensitivity_results.append({
        'Risk Aversion': ra,
        'Sharpe Ratio': test_metrics.sharpe_ratio,
        'Total Return': test_metrics.total_return,
        'Volatility': test_metrics.annual_volatility,
        'Max Drawdown': test_metrics.max_drawdown,
        'IC': test_result.ic
    })
    
    print(f"Risk Aversion = {ra}:")
    print(f"  Sharpe:  {test_metrics.sharpe_ratio:>6.3f}")
    print(f"  Return:  {test_metrics.total_return:>6.2%}")
    print(f"  Vol:     {test_metrics.annual_volatility:>6.2%}")
    print(f"  Max DD:  {test_metrics.max_drawdown:>6.2%}")
    print()

# Visualize sensitivity
sens_df = pd.DataFrame(sensitivity_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sharpe vs Risk Aversion
axes[0].plot(sens_df['Risk Aversion'], sens_df['Sharpe Ratio'], 
             marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0].set_title('Sharpe Ratio vs Risk Aversion', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Risk Aversion (λ)', fontsize=10)
axes[0].set_ylabel('Sharpe Ratio', fontsize=10)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3)

# Return vs Risk Aversion
axes[1].plot(sens_df['Risk Aversion'], sens_df['Total Return'], 
             marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_title('Total Return vs Risk Aversion', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Risk Aversion (λ)', fontsize=10)
axes[1].set_ylabel('Total Return', fontsize=10)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[1].grid(True, alpha=0.3)

# Max Drawdown vs Risk Aversion
axes[2].plot(sens_df['Risk Aversion'], sens_df['Max Drawdown'], 
             marker='^', linewidth=2, markersize=8, color='red')
axes[2].set_title('Max Drawdown vs Risk Aversion', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Risk Aversion (λ)', fontsize=10)
axes[2].set_ylabel('Max Drawdown', fontsize=10)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print("  • Higher risk aversion → Lower positions → Lower returns but also lower risk")
print("  • Optimal risk aversion depends on investor's risk tolerance")
print("  • Sharpe ratio typically peaks at moderate risk aversion (1.0-2.0)")

---

## Part 13: Summary and Key Insights

### What We've Accomplished

1. ✅ **Understood carry trading**: Economic rationale and why it works
2. ✅ **Calculated carry signals**: From calendar spreads (front - back)
3. ✅ **Generated alphas**: Using IC × Vol × Z (Grinold-Kahn formula)
4. ✅ **Estimated risk**: Covariance matrix with Ledoit-Wolf shrinkage
5. ✅ **Optimized portfolio**: Mean-variance optimization
6. ✅ **Backtested strategy**: End-to-end pipeline
7. ✅ **Analyzed performance**: TearSheet metrics and visualizations

### Key Performance Metrics

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Sharpe Ratio | See above | Risk-adjusted returns |
| Information Coefficient | See above | Signal quality |
| Total Return | See above | Cumulative return |
| Max Drawdown | See above | Downside risk |

### Economic Insights

1. **Carry Risk Premium**: Investors demand compensation for holding positions
2. **Regime Dependency**: Performance varies with curve slope regimes
3. **Mean Reversion**: Calendar spreads revert to fair value
4. **Risk-Return Tradeoff**: Higher risk aversion reduces returns and risk

### Technical Architecture

**Pipeline**:
```
Market Data (prices)
    ↓
Carry Signal (calendar spreads → z-scores)
    ↓
Alpha Generator (IC × Vol × Z → expected returns)
    ↓
Risk Model (Ledoit-Wolf covariance)
    ↓
Optimizer (Mean-variance optimal weights)
    ↓
Portfolio (track positions and P&L)
    ↓
TearSheet (performance metrics and analysis)
```

**Key Components**:
- `CarrySignal`: Calculates annualized carry from calendar spreads
- `AlphaGenerator`: Converts z-scores to expected returns (IC × Vol × Z)
- `LedoitWolfShrinkage`: Robust covariance estimation
- `MeanVarianceOptimizer`: Quadratic programming for optimal weights
- `Backtest`: End-to-end pipeline executor
- `TearSheet`: Comprehensive performance analysis

### MVP Philosophy

**Remember**: The goal is **ACCURATE MEASUREMENT**, not necessarily profitability.

- Negative Sharpe? That's data! Strategy doesn't work with these parameters.
- Negative IC? Signal is backwards! Should invert or abandon.
- High drawdown? Risk management needs improvement.

**All results are valuable for learning and iteration.**

---

## Next Steps

### Immediate Extensions

1. **Real Data**: Replace mock data with actual SOFR futures prices
2. **Transaction Costs**: Add bid-ask spreads and impact costs
3. **Multiple Signals**: Combine carry with momentum, mean reversion
4. **Advanced Optimization**: Add DV01 constraints, turnover limits

### Advanced Topics

1. **Dynamic IC**: Use time-varying Information Coefficient
2. **Regime Detection**: Adapt to market regimes (contango vs backwardation)
3. **Factor Models**: 3-factor PCA (level, slope, curvature)
4. **Machine Learning**: Enhance signals with ML predictions

### Other Notebooks

- `01_getting_started.ipynb`: Introduction to ARBS framework
- `02_strategy_comparison.ipynb`: Compare multiple strategies
- `03_parameter_tuning.ipynb`: Optimize parameters systematically
- `04_results_analysis.ipynb`: Advanced performance analytics
- `05_cross_asset_integration.ipynb`: Multi-asset strategies

### Documentation

- `docs/GRINOLD_KAHN_FRAMEWORK.md`: Complete framework documentation
- `docs/ADDING_CUSTOM_COMPONENTS.md`: Extend with custom signals
- `examples/run_backtest.py`: Python script version

---

**Happy Trading!** 📈